In [1]:
!pip install --upgrade pyarrow --break-system-packages

In [2]:
!pip install -q -U transformers accelerate datasets trl peft bitsandbytes

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
).to("cuda")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [4]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")
        break
else:
    print("All weights clean")

All weights clean


In [5]:
messages = [{"role": "user", "content": "Explain Direct Preference Optimisation in one sentence."}]

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,
    add_generation_prompt=True,
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain Direct Preference Optimisation in one sentence.
assistant
Direct preference optimisation is the process of maximizing profit or efficiency through strategic allocation and prioritization of resources based on direct preferences or objectives.


In [6]:
print(tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt=True,
))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain Direct Preference Optimisation in one sentence.<|im_end|>
<|im_start|>assistant



In [7]:
messages = [
    {"role": "system", "content": "You are a concise ML researcher."},
    {"role": "user", "content": "Explain Direct Preference Optimization in one sentence."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,

    add_generation_prompt=True,
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

system
You are a concise ML researcher.
user
Explain Direct Preference Optimization in one sentence.
assistant
Direct preference optimization involves identifying the most preferred option from among alternatives, often through a series of decision-making steps and iterative refinement to achieve optimal outcomes.


In [8]:
from datasets import load_dataset

ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

print(ds[0])

{'chosen': [{'content': 'Use the pygame library to write a version of the classic game Snake, with a unique twist', 'role': 'user'}, {'content': "Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:\n\n1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.\n2. Next, we'll create the game's grid, which will be used to represent the game board. We'll need to define the size of the grid and the spaces within it.\n3. After that, we'll create the snake object, which will be used to represent the player's movement. We'll need to define the size of the snake and the speed at which it moves.\n4. We'll also need to create a food object, which will be used to represent the food that the player must collect to score points. We'll need to define the location of the food and the speed at which it moves.\n5. Once we have these objects set up,

In [9]:
print(ds[1])

{'chosen': [{'content': "QUESTION: She was a horrible pet owner, she would put a what on her cat?\nOptions:\n- leave outside\n- sharp teeth\n- get wet\n- wool sweater\n- eat vegetables\nANSWER: Wool sweater can be put on a cat. Wool sweater is a vest. Horrible pet owners put on wool sweaters.\nThe answer is wool sweater\n\nQUESTION: Alcohol wasn't allowed at the meeting, so where did the alcoholic put his flask?\nOptions:\n- science lab\n- laboratory\n- coat pocket\n- the meeting room\n- chemistry lab\nANSWER: Person mostly wears coat when he is required to attend meeting and coat have pockets. Alcoholic person keeps alcohol flask with him which he can put in pocket of coat. Alcohol flask are easy to carry and conceal in pocket.\nThe answer is coat pocket\n\nQUESTION: How can a company get things to their customers?\nOptions:\n- mail order\n- carrier pigeon\n- own factory\n- ship goods\n- commit crime\nANSWER: To ship means to send goods to places far away. To send the customers the go

In [10]:
print(ds[1]["chosen"][0])

{'content': "QUESTION: She was a horrible pet owner, she would put a what on her cat?\nOptions:\n- leave outside\n- sharp teeth\n- get wet\n- wool sweater\n- eat vegetables\nANSWER: Wool sweater can be put on a cat. Wool sweater is a vest. Horrible pet owners put on wool sweaters.\nThe answer is wool sweater\n\nQUESTION: Alcohol wasn't allowed at the meeting, so where did the alcoholic put his flask?\nOptions:\n- science lab\n- laboratory\n- coat pocket\n- the meeting room\n- chemistry lab\nANSWER: Person mostly wears coat when he is required to attend meeting and coat have pockets. Alcoholic person keeps alcohol flask with him which he can put in pocket of coat. Alcohol flask are easy to carry and conceal in pocket.\nThe answer is coat pocket\n\nQUESTION: How can a company get things to their customers?\nOptions:\n- mail order\n- carrier pigeon\n- own factory\n- ship goods\n- commit crime\nANSWER: To ship means to send goods to places far away. To send the customers the goods, the com

In [11]:
ds

Dataset({
    features: ['chosen', 'rejected', 'score_chosen', 'score_rejected'],
    num_rows: 62135
})

In [12]:
ds = ds.remove_columns(["score_chosen", "score_rejected"])
print(ds.column_names)
print(len(ds))

['chosen', 'rejected']
62135


In [13]:
def tokenize_conversation(conversation, tokenizer, max_length=512):
   # conversation is the list of dicts
    # Tokenize just the prompt with
    prompt_result = tokenizer.apply_chat_template(
        [conversation[0]],
        add_generation_prompt=True,
    )
    prompt_ids = prompt_result["input_ids"]
    prompt_len = len(prompt_ids)

    # Tokenize full conversation
    full_result = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=False,
        max_length=max_length,
        truncation=True,
    )
    full_ids = full_result["input_ids"]

    # Labels: -100 for prompt, real ids for response
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    labels = labels[:len(full_ids)]

    return {
        "input_ids": full_ids,
        "labels": labels,
    }

In [14]:
result = tokenize_conversation(ds[0]["chosen"], tokenizer)
print("Input length:", len(result["input_ids"]))
print("Labels length:", len(result["labels"]))
print("Prompt tokens (masked):", result["labels"][:10])  # should be all -100
print("Response tokens (real):", result["labels"][-10:])

Input length: 512
Labels length: 512
Prompt tokens (masked): [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Response tokens (real): [220, 17, 20, 20, 11, 220, 15, 692, 2, 18614]


In [15]:
ref_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
).to("cuda")

ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [16]:
def compute_log_probs(model, input_ids, labels):
    outputs = model(input_ids=input_ids)
    logits = outputs.logits

    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    log_probs = torch.nn.functional.log_softmax(shift_logits.float(), dim=-1)
    log_probs_total = torch.tensor(0.0, device=input_ids.device)
    for token in range(len(shift_labels[0])):
      if shift_labels[0][token] == -100:
        continue
      else:
        log_probs_total += log_probs[0][token][shift_labels[0][token]]
    return log_probs_total


In [17]:
result = tokenize_conversation(ds[0]["chosen"], tokenizer)
input_ids = torch.tensor([result["input_ids"]], device="cuda")
labels = torch.tensor([result["labels"]], device="cuda")

with torch.no_grad():
    logp = compute_log_probs(ref_model, input_ids, labels)
    print(f"Log-prob: {logp.item():.2f}")

Log-prob: -413.43


In [18]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps,
             beta=0.1):
    chosen_advantage  = policy_chosen_logps - ref_chosen_logps
    rejected_advantage = policy_rejected_logps - ref_rejected_logps

    logits = beta * (chosen_advantage - rejected_advantage)
    loss = -torch.nn.functional.logsigmoid(logits)

    reward_margin = (chosen_advantage - rejected_advantage).detach()

    return loss, reward_margin

In [19]:
from torch.optim import AdamW

beta = 0.1
lr = 5e-7
num_episodes = 500
max_length = 512

run_loss = 0
optimizer = AdamW(model.parameters(), lr=lr)
model.train()
for episode in range(num_episodes):
  convo_chosen = tokenize_conversation(ds[episode]["chosen"],tokenizer,max_length)
  convo_rejected = tokenize_conversation(ds[episode]["rejected"],tokenizer,max_length)
  if all(l == -100 for l in convo_chosen["labels"]) or all(l == -100 for l in convo_rejected["labels"]):
    continue
  chosen_ids  = torch.tensor([convo_chosen["input_ids"]], device="cuda")
  chosen_lab  = torch.tensor([convo_chosen["labels"]], device="cuda")
  rejected_ids = torch.tensor([convo_rejected["input_ids"]], device="cuda")
  rejected_lab = torch.tensor([convo_rejected["labels"]], device="cuda")
  with torch.no_grad():
    ref_chosen_logp   = compute_log_probs(ref_model, chosen_ids, chosen_lab)
    ref_rejected_logp = compute_log_probs(ref_model, rejected_ids, rejected_lab)
  chosen_logp = compute_log_probs(model,chosen_ids,chosen_lab)
  rejected_logp = compute_log_probs(model, rejected_ids, rejected_lab)
  loss,reward_margin = dpo_loss(chosen_logp,rejected_logp,ref_chosen_logp,ref_rejected_logp,beta)
  run_loss += loss.item()
  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  optimizer.step()

  if (episode+1) % 50 == 0:
    print(f"Loss: {run_loss/50}")
    run_loss = 0


Loss: 0.667579448223114
Loss: 0.6532892179489136
Loss: 0.6526021873950958
Loss: 1.2440483355522156
Loss: 1.2913844764232636
Loss: 0.5865543079376221
Loss: 0.6391893243789672
Loss: 0.6672866547107696


In [22]:
model.eval()
messages = [{"role": "user", "content": "Explain Direct Preference Optimisation in one sentence."}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", return_dict=True, add_generation_prompt=True).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.7)
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain Direct Preference Optimisation in one sentence.
assistant
Direct preference optimisation is a strategy that involves identifying and prioritizing the most critical preferences or needs of users to enhance user experience directly without relying on third-party services.
